# ValueCausalLoss + GPT-2 Model Testing

This notebook demonstrates:
1. Loading training configuration from `configs/training/trigo-value-gpt2.yaml`
2. Creating TGNValueDataset and dataloader
3. Initializing GPT-2 model with ValueCausalLoss wrapper (dual-head: policy + value)
4. Forward pass through model with one batch
5. Backward pass (loss backpropagation)
6. Inspecting gradients and model outputs
7. Analyzing policy and value predictions separately

## 1. Environment Setup

In [2]:
import sys
import os

sys.path.append('..')
os.chdir('..')

In [3]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from omegaconf import OmegaConf

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

from trigor.data import TGNValueDataset
from trigor.models import make_model

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Registered dataset: TGNDataset
Registered dataset: TGNValueDataset


/home/camus/work/trigoRL/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Registered model: GPT2CausalLM
Registered model: LlamaCausalLM
Registered model: RwkvCausalLM
Registered model: xLSTMCausalLM
Registered model: AttentionCausalLoss
Registered model: ValueCausalLoss
Registered model: evaluation
Project root: /home/camus/work/trigoRL
PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3090


## 2. Load Configuration

Load the ValueCausalLoss training configuration from YAML file.

In [4]:
from datetime import datetime

# Load configuration
config_path = project_root / "configs/training/trigo-value-gpt2.yaml"
cfg = OmegaConf.load(config_path)

# Register custom OmegaConf resolver for date
OmegaConf.register_new_resolver("date", lambda: datetime.now().strftime("%Y%m%d"))

# Register custom resolver to remove .local suffix from config names
OmegaConf.register_new_resolver(
    "remove_local_suffix",
    lambda s: s[:-6] if s.endswith('.local') else s
)

# Register Hydra resolvers for notebook environment
def hydra_resolver(path: str) -> str:
    """Mock Hydra resolver for notebook environment."""
    if path == "job.config_name":
        return config_path.stem  # Returns "trigo-value-gpt2"
    elif path == "runtime.cwd":
        return str(project_root)
    else:
        return ""

OmegaConf.register_new_resolver("hydra", hydra_resolver)

# Resolve paths
OmegaConf.update(cfg, "paths.root", str(project_root))
OmegaConf.resolve(cfg)

print("Configuration loaded successfully!")
print("\nModel configuration:")
print(f"  Type: {cfg.model.type}")
print(f"  Base model: {cfg.model.config.model_config.type}")
print(f"  Hidden size: {cfg.model.config.model_config.config.hidden_size}")
print(f"  Num layers: {cfg.model.config.model_config.config.num_layers}")
print(f"  Num heads: {cfg.model.config.model_config.config.num_heads}")
print(f"  Vocab size: {cfg.model.config.model_config.config.vocab_size}")
print(f"  Max seq len: {cfg.model.config.model_config.config.max_seq_len}")

print("\n  Value head config:")
print(f"    Hidden dim: {cfg.model.config.value_head_config.hidden_dim}")
print(f"    Intermediate dim: {cfg.model.config.value_head_config.intermediate_dim}")
print(f"    Bottleneck dim: {cfg.model.config.value_head_config.bottleneck_dim}")

print("\n  Loss configuration:")
print(f"    Lambda policy: {cfg.model.config.lambda_policy}")
print(f"    Lambda value: {cfg.model.config.lambda_value}")
print(f"    Gamma (discount): {cfg.model.config.gamma}")
print(f"    Territory value factor: {cfg.model.config.territory_value_factor}")
print(f"    Ignore index: {cfg.model.config.ignore_index}")
print(f"    Label smoothing: {cfg.model.config.label_smoothing}")

print("\nData configuration:")
print(f"  Type: {cfg.data.type}")
print(f"  Data dir: {cfg.data.data_dir}")
print(f"  Max length: {cfg.data.max_length}")
print(f"  Batch size: {cfg.data.loader.batch_size}")

print("\nTraining configuration:")
print(f"  Learning rate: {cfg.training.learning_rate}")
print(f"  Weight decay: {cfg.training.weight_decay}")
print(f"  Max grad norm: {cfg.training.max_grad_norm}")
print(f"  Dtype: {cfg.training.dtype}")

Configuration loaded successfully!

Model configuration:
  Type: ValueCausalLoss
  Base model: GPT2CausalLM
  Hidden size: 256
  Num layers: 6
  Num heads: 8
  Vocab size: 128
  Max seq len: 1024

  Value head config:
    Hidden dim: 256
    Intermediate dim: 512
    Bottleneck dim: 64

  Loss configuration:
    Lambda policy: 1.0
    Lambda value: 0.5
    Gamma (discount): 0.99
    Territory value factor: 1.0
    Ignore index: 0
    Label smoothing: 0.1

Data configuration:
  Type: TGNValueDataset
  Data dir: /home/camus/work/trigoRL/third_party/trigo/trigo-web/tools/output/selfplay
  Max length: 1024
  Batch size: 1

Training configuration:
  Learning rate: 0.0001
  Weight decay: 0.01
  Max grad norm: 1.0
  Dtype: bfloat16


## 3. Load Dataset

Create TGNValueDataset and inspect some samples. This dataset includes value_score and move_end_positions.

In [5]:
# Create dataset
print("Loading TGNValueDataset...")
dataset = TGNValueDataset.from_config(cfg.data)

print(f"\nDataset loaded successfully!")
print(f"  Total samples: {len(dataset)}")
print(f"  Tokenizer vocab size: {dataset.tokenizer.get_vocab_size()}")
print(f"  Max length: {dataset.max_length}")
print(f"  PAD token ID: {dataset.tokenizer.PAD_ID}")
print(f"  START token ID: {dataset.tokenizer.START_ID}")
print(f"  END token ID: {dataset.tokenizer.END_ID}")
print(f"  VALUE token ID: {dataset.tokenizer.VALUE_ID}")

Loading TGNValueDataset...
Loaded 93 TGN files from /home/camus/work/trigoRL/third_party/trigo/trigo-web/tools/output/selfplay

Dataset loaded successfully!
  Total samples: 93
  Tokenizer vocab size: 128
  Max length: 1024
  PAD token ID: 0
  START token ID: 1
  END token ID: 2
  VALUE token ID: 3


In [6]:
# Inspect a single sample
print("\n" + "="*80)
print("Sample Inspection")
print("="*80)

sample = dataset[0]
print(f"\nSample 0:")
print(f"  Input IDs shape: {sample['input_ids'].shape}")
print(f"  Labels shape: {sample['labels'].shape}")
print(f"  Attention mask shape: {sample['attention_mask'].shape}")
print(f"  Value score: {sample['value_score'].item():.2f}")
print(f"  Move end positions: {sample['move_end_positions'].tolist()}")
print(f"  Number of moves: {len(sample['move_end_positions'])}")

print(f"\n  First 20 input tokens: {sample['input_ids'][:20].tolist()}")
print(f"  First 20 labels: {sample['labels'][:20].tolist()}")
print(f"  First 20 attention mask: {sample['attention_mask'][:20].tolist()}")

# Count non-padding tokens
non_pad_count = (sample['attention_mask'] == 1).sum().item()
print(f"\n  Non-padding tokens: {non_pad_count} / {sample['input_ids'].shape[0]}")


Sample Inspection

Sample 0:
  Input IDs shape: torch.Size([1023])
  Labels shape: torch.Size([1023])
  Attention mask shape: torch.Size([1023])
  Value score: 4.00
  Move end positions: [19, 22, 28, 31, 37, 40, 46, 49, 55, 58, 64, 67, 73, 76, 84]
  Number of moves: 15

  First 20 input tokens: [1, 91, 66, 111, 97, 114, 100, 32, 53, 120, 51, 93, 10, 10, 49, 46, 32, 98, 97, 32]
  First 20 labels: [91, 66, 111, 97, 114, 100, 32, 53, 120, 51, 93, 10, 10, 49, 46, 32, 98, 97, 32, 98]
  First 20 attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

  Non-padding tokens: 91 / 1023


## 4. Create DataLoader

Create dataloader with collate function for batching. Note that TGNValueDataset needs special collation for move_end_positions.

In [7]:
# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=cfg.data.loader.batch_size,
    shuffle=cfg.data.loader.shuffle,
    num_workers=0,  # Use 0 for notebook to avoid multiprocessing issues
    collate_fn=TGNValueDataset.collate_batch,
    pin_memory=False,  # Disable for CPU testing
)

print(f"DataLoader created successfully!")
print(f"  Batch size: {cfg.data.loader.batch_size}")
print(f"  Total batches: {len(dataloader)}")
print(f"  Shuffle: {cfg.data.loader.shuffle}")

DataLoader created successfully!
  Batch size: 1
  Total batches: 93
  Shuffle: True


In [8]:
# Get one batch for testing
batch = next(iter(dataloader))

print("\n" + "="*80)
print("Batch Inspection")
print("="*80)

print(f"\nBatch shapes:")
print(f"  Input IDs: {batch['input_ids'].shape}")
print(f"  Labels: {batch['labels'].shape}")
print(f"  Attention mask: {batch['attention_mask'].shape}")
print(f"  Value score: {batch['value_score'].shape}")
print(f"  Move end positions: list of {len(batch['move_end_positions'])} tensors")

print(f"\nBatch statistics:")
print(f"  Total tokens: {batch['input_ids'].numel()}")
print(f"  Valid tokens: {(batch['attention_mask'] == 1).sum().item()}")
print(f"  Padding tokens: {(batch['attention_mask'] == 0).sum().item()}")
print(f"  Padding ratio: {(batch['attention_mask'] == 0).sum().item() / batch['input_ids'].numel():.2%}")

# Show first sample in batch
print(f"\nFirst sample in batch:")
print(f"  Value score: {batch['value_score'][0].item():.2f}")
print(f"  Number of moves: {len(batch['move_end_positions'][0])}")
print(f"  Move end positions: {batch['move_end_positions'][0].tolist()}")
print(f"  First 20 tokens: {batch['input_ids'][0, :20].tolist()}")
print(f"  Valid length: {(batch['attention_mask'][0] == 1).sum().item()}")


Batch Inspection

Batch shapes:
  Input IDs: torch.Size([1, 1023])
  Labels: torch.Size([1, 1023])
  Attention mask: torch.Size([1, 1023])
  Value score: torch.Size([1])
  Move end positions: list of 1 tensors

Batch statistics:
  Total tokens: 1023
  Valid tokens: 122
  Padding tokens: 901
  Padding ratio: 88.07%

First sample in batch:
  Value score: -5.00
  Number of moves: 18
  Move end positions: [22, 26, 33, 37, 44, 48, 55, 59, 66, 70, 77, 81, 88, 92, 99, 103, 110, 115]
  First 20 tokens: [1, 91, 66, 111, 97, 114, 100, 32, 50, 120, 51, 120, 51, 93, 10, 10, 49, 46, 32, 97]
  Valid length: 122


## 5. Create Model

Initialize GPT-2 model with ValueCausalLoss wrapper (dual-head: policy + value).

In [9]:
# Create model using factory
print("Creating model...")
model = make_model(cfg.model.type, cfg.model.config)

print("\nModel created successfully!")
print(model)

# Count parameters
params = model.count_parameters()
print(f"\nParameter counts:")
print(f"  Total: {params['total']:,}")
print(f"  Trainable: {params['trainable']:,}")
print(f"  Non-trainable: {params['non_trainable']:,}")
print(f"\n  Base model: {params['base_model']['total']:,}")
print(f"  Value head: {params['value_head']['total']:,}")

# Get model info
info = model.get_model_info()
print(f"\nModel info:")
for key, value in info.items():
    if key not in ['base_model_info', 'value_head_info']:
        print(f"  {key}: {value}")

Creating model...

Model created successfully!
ValueCausalLoss(
  model_type=GPT2CausalLM,
  total_parameters=5,199,617,
  base_model_parameters=5,033,984,
  value_head_parameters=165,633,
  lambda_policy=1.0,
  lambda_value=0.5,
  gamma=0.99,
  territory_value_factor=1.0
)

Parameter counts:
  Total: 5,199,617
  Trainable: 5,199,617
  Non-trainable: 0

  Base model: 5,033,984
  Value head: 165,633

Model info:
  model_type: GPT2CausalLM
  lambda_policy: 1.0
  lambda_value: 0.5
  gamma: 0.99
  territory_value_factor: 1.0
  ignore_index: 0
  label_smoothing: 0.1
  value_id: 3
  pad_id: 0


In [10]:
# Convert model to specified dtype (same as trainer does)
dtype_map = {
    'float32': torch.float32,
    'fp32': torch.float32,
    'float16': torch.float16,
    'fp16': torch.float16,
    'bfloat16': torch.bfloat16,
    'bf16': torch.bfloat16,
}

dtype_str = cfg.training.dtype
dtype = dtype_map.get(dtype_str.lower(), torch.float32)

print(f"\nConverting model to dtype: {dtype}")
print(f"  Before conversion: {next(model.parameters()).dtype}")

if dtype != torch.float32:
    model = model.to(dtype=dtype)

print(f"  After conversion: {next(model.parameters()).dtype}")


Converting model to dtype: torch.bfloat16
  Before conversion: torch.float32
  After conversion: torch.bfloat16


## 6. Forward Pass

Run forward pass through the model with one batch. This will compute both policy and value losses.

In [11]:
# Set model to training mode
model.train()

print("\n" + "="*80)
print("Forward Pass")
print("="*80)

# Forward pass - pass entire batch dict
print("\nRunning forward pass...")
outputs = model(batch, return_logits=True)


Forward Pass

Running forward pass...


In [12]:
print("\nForward pass completed!")
print(f"\nOutput keys: {list(outputs.keys())}")

print(f"\nOutput shapes:")
print(f"  Loss (total): {outputs['loss'].shape} (scalar)")
print(f"  Policy loss: {outputs['policy_loss'].shape} (scalar)")
print(f"  Value loss: {outputs['value_loss'].shape} (scalar)")
print(f"  Policy error: {outputs['policy_error'].shape} (scalar)")
print(f"  Value MAE: {outputs['value_mae'].shape} (scalar)")
print(f"  Value MSE: {outputs['value_mse'].shape} (scalar)")
print(f"  Num policy tokens: {outputs['num_policy_tokens'].shape} (scalar)")
print(f"  Num value predictions: {outputs['num_value_predictions'].shape} (scalar)")
print(f"  Logits: {outputs['logits'].shape} [batch_size, extended_seq_len, vocab_size]")
print(f"  Value predictions: {outputs['value_predictions'].shape} [total_value_tokens]")

print(f"\nMetrics:")
print(f"  Total Loss: {outputs['loss'].item():.4f}")
print(f"    Policy Loss: {outputs['policy_loss'].item():.4f} (weight: {cfg.model.config.lambda_policy})")
print(f"    Value Loss: {outputs['value_loss'].item():.4f} (weight: {cfg.model.config.lambda_value})")
print(f"  Policy Error: {outputs['policy_error'].item():.4f} ({outputs['policy_error'].item()*100:.2f}%)")
print(f"  Value MAE: {outputs['value_mae'].item():.4f}")
print(f"  Value MSE: {outputs['value_mse'].item():.4f}")
print(f"  Policy tokens: {outputs['num_policy_tokens'].item()}")
print(f"  Value predictions: {outputs['num_value_predictions'].item()}")

# Analyze logits
logits = outputs['logits']
print(f"\nLogits statistics:")
print(f"  Shape: {logits.shape}")
print(f"  Dtype: {logits.dtype}")
print(f"  Min: {logits.min().item():.4f}")
print(f"  Max: {logits.max().item():.4f}")
print(f"  Mean: {logits.mean().item():.4f}")
print(f"  Std: {logits.std().item():.4f}")

# Analyze value predictions
value_preds = outputs['value_predictions']
print(f"\nValue predictions statistics:")
print(f"  Shape: {value_preds.shape}")
print(f"  Dtype: {value_preds.dtype}")
print(f"  Min: {value_preds.min().item():.4f}")
print(f"  Max: {value_preds.max().item():.4f}")
print(f"  Mean: {value_preds.mean().item():.4f}")
print(f"  Std: {value_preds.std().item():.4f}")
print(f"  First 10 predictions: {value_preds[:10].tolist()}")

# Show value target computation (matching the formula in valueCausalLoss.py)
print(f"\nValue target computation (for first sample):")
score = batch['value_score'][0]
sign = torch.sgn(score).item()
log_abs_score = torch.log(torch.abs(score)).item()
target = sign * (1 + log_abs_score) * cfg.model.config.territory_value_factor
print(f"  Score: {score.item():.2f}")
print(f"  Sign (sgn): {sign}")
print(f"  Log(|score|): {log_abs_score:.4f}")
print(f"  Target formula: sign * (1 + log(|score|)) * factor")
print(f"  Target (undiscounted): {target:.4f}")
print(f"  Discount factor (gamma): {cfg.model.config.gamma}")
print(f"  Example: {sign} * (1 + {log_abs_score:.4f}) * {cfg.model.config.territory_value_factor} = {target:.4f}")


Forward pass completed!

Output keys: ['loss', 'policy_loss', 'value_loss', 'policy_error', 'value_mae', 'value_mse', 'num_policy_tokens', 'num_value_predictions', 'logits', 'value_predictions']

Output shapes:
  Loss (total): torch.Size([]) (scalar)
  Policy loss: torch.Size([]) (scalar)
  Value loss: torch.Size([]) (scalar)
  Policy error: torch.Size([]) (scalar)
  Value MAE: torch.Size([]) (scalar)
  Value MSE: torch.Size([]) (scalar)
  Num policy tokens: torch.Size([]) (scalar)
  Num value predictions: torch.Size([]) (scalar)
  Logits: torch.Size([1, 1024, 128]) [batch_size, extended_seq_len, vocab_size]
  Value predictions: torch.Size([1]) [total_value_tokens]

Metrics:
  Total Loss: 7.5625
    Policy Loss: 4.7500 (weight: 1.0)
    Value Loss: 5.6250 (weight: 0.5)
  Policy Error: 0.9008 (90.08%)
  Value MAE: 2.3750
  Value MSE: 5.6250
  Policy tokens: 121
  Value predictions: 1

Logits statistics:
  Shape: torch.Size([1, 1024, 128])
  Dtype: torch.bfloat16
  Min: -1.4219
  Max: 1

/tmp/ipykernel_1846373/1653958664.py:44: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  print(f"  Std: {value_preds.std().item():.4f}")


## 7. Backward Pass

Perform backward pass and inspect gradients.

In [13]:
print("\n" + "="*80)
print("Backward Pass")
print("="*80)

# Get loss
loss = outputs['loss']
print(f"\nLoss before backward: {loss.item():.4f}")
print(f"Loss requires_grad: {loss.requires_grad}")

# Zero gradients (simulating optimizer.zero_grad())
model.zero_grad()
print("\nGradients zeroed.")

# Backward pass
print("\nRunning backward pass...")
loss.backward()
print("Backward pass completed!")


Backward Pass

Loss before backward: 7.5625
Loss requires_grad: True

Gradients zeroed.

Running backward pass...
Backward pass completed!


## 8. Inspect Gradients

Check that gradients have been computed correctly for both base model and value head.

In [14]:
print("\n" + "="*80)
print("Gradient Inspection")
print("="*80)

# Collect gradient statistics
grad_stats = []
total_grad_norm = 0.0
params_with_grad = 0
params_without_grad = 0

base_model_grad_norm = 0.0
value_head_grad_norm = 0.0

for name, param in model.named_parameters():
    if param.requires_grad:
        if param.grad is not None:
            grad = param.grad
            grad_norm = grad.norm().item()
            total_grad_norm += grad_norm ** 2
            params_with_grad += 1
            
            # Track base model vs value head separately
            if 'value_head' in name:
                value_head_grad_norm += grad_norm ** 2
            else:
                base_model_grad_norm += grad_norm ** 2
            
            grad_stats.append({
                'name': name,
                'shape': list(param.shape),
                'grad_norm': grad_norm,
                'grad_min': grad.min().item(),
                'grad_max': grad.max().item(),
                'grad_mean': grad.mean().item(),
                'grad_std': grad.std().item(),
                'is_value_head': 'value_head' in name,
            })
        else:
            params_without_grad += 1

total_grad_norm = total_grad_norm ** 0.5
base_model_grad_norm = base_model_grad_norm ** 0.5
value_head_grad_norm = value_head_grad_norm ** 0.5

print(f"\nGradient statistics:")
print(f"  Parameters with gradients: {params_with_grad}")
print(f"  Parameters without gradients: {params_without_grad}")
print(f"  Total gradient norm: {total_grad_norm:.4f}")
print(f"    Base model gradient norm: {base_model_grad_norm:.4f}")
print(f"    Value head gradient norm: {value_head_grad_norm:.4f}")

# Show top 5 base model layers by gradient norm
print(f"\nTop 5 base model layers by gradient norm:")
base_stats = [s for s in grad_stats if not s['is_value_head']]
sorted_base = sorted(base_stats, key=lambda x: x['grad_norm'], reverse=True)[:5]
for i, stat in enumerate(sorted_base, 1):
    print(f"\n  {i}. {stat['name']}")
    print(f"     Shape: {stat['shape']}")
    print(f"     Grad norm: {stat['grad_norm']:.4f}")
    print(f"     Grad range: [{stat['grad_min']:.4f}, {stat['grad_max']:.4f}]")

# Show all value head layers
print(f"\nValue head layers:")
value_stats = [s for s in grad_stats if s['is_value_head']]
for i, stat in enumerate(value_stats, 1):
    print(f"\n  {i}. {stat['name']}")
    print(f"     Shape: {stat['shape']}")
    print(f"     Grad norm: {stat['grad_norm']:.4f}")
    print(f"     Grad range: [{stat['grad_min']:.4f}, {stat['grad_max']:.4f}]")


Gradient Inspection

Gradient statistics:
  Parameters with gradients: 86
  Parameters without gradients: 0
  Total gradient norm: 347.0866
    Base model gradient norm: 343.0210
    Value head gradient norm: 52.9692

Top 5 base model layers by gradient norm:

  1. model.transformer.h.0.mlp.c_proj.weight
     Shape: [1024, 256]
     Grad norm: 148.0000
     Grad range: [-3.7188, 3.1094]

  2. model.transformer.h.0.attn.c_proj.weight
     Shape: [256, 256]
     Grad norm: 137.0000
     Grad range: [-6.8438, 8.5000]

  3. model.transformer.h.1.mlp.c_proj.weight
     Shape: [1024, 256]
     Grad norm: 111.5000
     Grad range: [-2.9219, 3.7969]

  4. model.transformer.h.2.mlp.c_proj.weight
     Shape: [1024, 256]
     Grad norm: 90.0000
     Grad range: [-2.7500, 3.5312]

  5. model.transformer.h.1.attn.c_proj.weight
     Shape: [256, 256]
     Grad norm: 87.0000
     Grad range: [-2.9219, 2.8125]

Value head layers:

  1. value_head.fc1.weight
     Shape: [512, 256]
     Grad norm: 29.7

/tmp/ipykernel_1846373/1074943468.py:35: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  'grad_std': grad.std().item(),


## 9. Gradient Clipping

Demonstrate gradient clipping as used in training.

In [15]:
print("\n" + "="*80)
print("Gradient Clipping")
print("="*80)

print(f"\nGradient norm before clipping: {total_grad_norm:.4f}")
print(f"Max gradient norm (from config): {cfg.training.max_grad_norm}")

# Clip gradients
torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.max_grad_norm)

# Recompute gradient norm after clipping
new_grad_norm = 0.0
for param in model.parameters():
    if param.grad is not None:
        new_grad_norm += param.grad.norm().item() ** 2
new_grad_norm = new_grad_norm ** 0.5

print(f"Gradient norm after clipping: {new_grad_norm:.4f}")

if total_grad_norm > cfg.training.max_grad_norm:
    print(f"\n✓ Gradients were clipped (reduced by {(1 - new_grad_norm/total_grad_norm)*100:.2f}%)")
else:
    print(f"\n✓ No clipping needed (gradient norm within limit)")


Gradient Clipping

Gradient norm before clipping: 347.0866
Max gradient norm (from config): 1.0
Gradient norm after clipping: 0.9968

✓ Gradients were clipped (reduced by 99.71%)


## 10. Optimizer Step

Demonstrate optimizer step without actually updating parameters.

In [16]:
print("\n" + "="*80)
print("Optimizer Setup")
print("="*80)

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
)

print(f"\nOptimizer: AdamW")
print(f"  Learning rate: {cfg.training.learning_rate}")
print(f"  Weight decay: {cfg.training.weight_decay}")
print(f"  Number of parameter groups: {len(optimizer.param_groups)}")

# Save a parameter value before update
first_param = next(model.parameters())
param_before = first_param.data.clone()
print(f"\nFirst parameter stats before update:")
print(f"  Mean: {param_before.mean().item():.6f}")
print(f"  Std: {param_before.std().item():.6f}")

# Perform optimizer step
print("\nPerforming optimizer step...")
optimizer.step()
print("Optimizer step completed!")

# Check parameter change
param_after = first_param.data
param_change = (param_after - param_before).abs().mean().item()
print(f"\nFirst parameter stats after update:")
print(f"  Mean: {param_after.mean().item():.6f}")
print(f"  Std: {param_after.std().item():.6f}")
print(f"  Average absolute change: {param_change:.8f}")

print(f"\n✓ Parameters have been updated!")


Optimizer Setup

Optimizer: AdamW
  Learning rate: 0.0001
  Weight decay: 0.01
  Number of parameter groups: 1

First parameter stats before update:
  Mean: -0.000030
  Std: 0.020020

Performing optimizer step...
Optimizer step completed!

First parameter stats after update:
  Mean: -0.000030
  Std: 0.020020
  Average absolute change: 0.00009966

✓ Parameters have been updated!


## 11. Summary

Summary of the test run.

In [17]:
print("\n" + "="*80)
print("TEST SUMMARY")
print("="*80)

print(f"\n✓ Configuration loaded from: {config_path.name}")
print(f"✓ Dataset loaded: {len(dataset)} samples (TGNValueDataset)")
print(f"✓ Model created: {params['total']:,} parameters")
print(f"    Base model: {params['base_model']['total']:,} parameters")
print(f"    Value head: {params['value_head']['total']:,} parameters")
print(f"✓ Forward pass successful (dual-head)")
print(f"✓ Backward pass successful")
print(f"✓ Gradients computed and clipped")
print(f"✓ Optimizer step completed")

print(f"\nMetrics from forward pass:")
print(f"  Total Loss: {outputs['loss'].item():.4f}")
print(f"  Policy Loss: {outputs['policy_loss'].item():.4f} (weight: {cfg.model.config.lambda_policy})")
print(f"  Value Loss: {outputs['value_loss'].item():.4f} (weight: {cfg.model.config.lambda_value})")
print(f"  Policy Error: {outputs['policy_error'].item()*100:.2f}%")
print(f"  Value MAE: {outputs['value_mae'].item():.4f}")

print(f"\nModel configuration:")
print(f"  Gamma (discount factor): {cfg.model.config.gamma}")
print(f"  Territory value factor: {cfg.model.config.territory_value_factor}")
print(f"  Lambda policy: {cfg.model.config.lambda_policy}")
print(f"  Lambda value: {cfg.model.config.lambda_value}")

print(f"\nModel is ready for training!")
print(f"\nNext steps:")
print(f"  1. Implement training loop over multiple epochs")
print(f"  2. Add validation evaluation")
print(f"  3. Add checkpointing and logging")
print(f"  4. Monitor both policy and value metrics")
print(f"  5. Optionally enable WandB tracking")


TEST SUMMARY

✓ Configuration loaded from: trigo-value-gpt2.yaml
✓ Dataset loaded: 93 samples (TGNValueDataset)
✓ Model created: 5,199,617 parameters
    Base model: 5,033,984 parameters
    Value head: 165,633 parameters
✓ Forward pass successful (dual-head)
✓ Backward pass successful
✓ Gradients computed and clipped
✓ Optimizer step completed

Metrics from forward pass:
  Total Loss: 7.5625
  Policy Loss: 4.7500 (weight: 1.0)
  Value Loss: 5.6250 (weight: 0.5)
  Policy Error: 90.08%
  Value MAE: 2.3750

Model configuration:
  Gamma (discount factor): 0.99
  Territory value factor: 1.0
  Lambda policy: 1.0
  Lambda value: 0.5

Model is ready for training!

Next steps:
  1. Implement training loop over multiple epochs
  2. Add validation evaluation
  3. Add checkpointing and logging
  4. Monitor both policy and value metrics
  5. Optionally enable WandB tracking
